Notebook examines parameter recapitulation using synthetic data.

# Setup

In [1]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-07-27 15:06:17.241174: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-27 15:06:18.141763: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-07-27 15:06:18.141788: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster()
client = Client(cluster)

# Ground truth carpentry 

In [31]:
ground_truth_mu=pd.read_csv("../../notebooks/demos/parameter_extraction_demo/synthetic_ground_truth.tsv",sep="\t")
#fix col names
ground_truth_mu=ground_truth_mu.rename({'CRE':'cre_id','mean':'true_mu','Cell-type':'cell_type'},axis=1)
#set reference CRE
ground_truth_mu.loc[
    ground_truth_mu["cre_id"].isin(["nobody","weak"]),
    "cre_id"
]="reference"
#set reference cell-type
ground_truth_mu.loc[
    ground_truth_mu["cell_type"]=="liver",
    "cell_type"
]="reference"
#set index
ground_truth_mu=ground_truth_mu.set_index(['cell_type','cre_id'])
#display
ground_truth_mu

true_mu
cell_type cre_id                
brain     reference            1
          mediumbody          10
          everybody          114
          redgene             30
          neurogene           99
          reference            5
          pos_control        201
          pos_control_B      102
          hepatogene          14
blood     reference            2
          mediumbody          12
          everybody          109
          redgene            112
          neurogene           16
          reference            2
          pos_control        222
          pos_control_B      103
          hepatogene          15
reference reference            3
          mediumbody          11
          everybody          110
          redgene             15
          neurogene           16
          reference            3
          pos_control        201
          pos_control_B      111
          hepatogene          66

# Parameter extraction

In [4]:
dat=scm.scMPRA_data.from_tsv("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres_v3.tsv")
dat.ortho_filter()
dat.set_negative_controls(["nobody","weak"])
dat.set_reference_cell("liver")

primordial=scm.ortho()
primordial.criss_cross(client=client,dat=dat)
primordial.extract_params(client)

batch=scm.simulation_batch(primordial)
batch.describe_primordial()

scMPRAforge: INFO: Dropped 0 of 27 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


# Preprocess by_cell_type

In [32]:
by_cell_type=batch.description_primordial_by_cell_type.compute()
by_cell_type=by_cell_type.set_index(['cell_type','cre_id'])
by_cell_type.head()

index rep_id          mu        zi     theta  cells  \
cell_type cre_id                                                            
blood     everybody       0      1  106.093429  0.100927  3.327685    489   
          everybody       1      2  106.093429  0.202659  3.327685    514   
          everybody       2      3  106.093429  0.306378  3.327685    496   
          hepatogene      3      1   14.828171  0.100927  3.327685    516   
          hepatogene      4      2   14.828171  0.202659  3.327685    523   

                             r  sigmasquare         p  
cell_type cre_id                                       
blood     everybody   3.327685  3488.569374  0.030412  
          everybody   3.327685  3488.569374  0.030412  
          everybody   3.327685  3488.569374  0.030412  
          hepatogene  3.327685    80.902520  0.183284  
          hepatogene  3.327685    80.902520  0.183284

# Preprocess by_cre

In [47]:
by_cre=batch.description_primordial_by_cre.compute()
by_cre=by_cre.set_index(['cell_type','cre_id'])
by_cre.head()

index rep_id          mu        zi     theta  cells  \
cell_type cre_id                                                           
blood     everybody      0      1  106.093745  0.092962  3.259602    489   
          everybody      1      2  106.093745  0.206867  3.259602    514   
          everybody      2      3  106.093745  0.305204  3.259602    496   
brain     everybody      3      1  112.408435  0.092962  3.259602    449   
          everybody      4      2  112.408435  0.206867  3.259602    451   

                            r  sigmasquare         p  
cell_type cre_id                                      
blood     everybody  3.259602  3559.240504  0.029808  
          everybody  3.259602  3559.240504  0.029808  
          everybody  3.259602  3559.240504  0.029808  
brain     everybody  3.259602  3988.850365  0.028181  
          everybody  3.259602  3988.850365  0.028181

# Join & compare

In [48]:
comp_by_cell_type=by_cell_type.join(ground_truth_mu)
comp_by_cell_type=comp_by_cell_type[["mu","true_mu"]].drop_duplicates()

comp_by_cre=by_cre.join(ground_truth_mu)
comp_by_cre=comp_by_cre[["mu","true_mu"]].drop_duplicates()

In [44]:
def mse(df):
    return np.mean((df["mu"]-df["true_mu"])**2)
mse(comp_by_cell_type)

2.767962728640191

In [49]:
mse(comp_by_cre)

3.387048380280456